In [2]:
import os
import sys
import time
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import autocast, GradScaler

In [22]:
# ==========================================
# 1. 加载配置文件
# ==========================================
def load_config(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    return config

# 假设配置文件名为 det_db_mv3.yaml
cfg = load_config('det_db_mv3.yaml') 
global_config = config['Global']
global_config

{'use_gpu': True,
 'epoch_num': 1200,
 'log_smooth_window': 20,
 'print_batch_step': 10,
 'save_model_dir': './output/db_mv3/',
 'save_epoch_step': 1200,
 'eval_batch_step': [0, 2000],
 'cal_metric_during_train': False,
 'pretrained_model': './pretrain_models/MobileNetV3_large_x0_5_pretrained',
 'checkpoints': None,
 'save_inference_dir': None,
 'use_visualdl': False,
 'infer_img': 'doc/imgs_en/img_10.jpg',
 'save_res_path': './output/det_db/predicts_db.txt'}

In [25]:
cfg

{'Global': {'use_gpu': True,
  'epoch_num': 1200,
  'log_smooth_window': 20,
  'print_batch_step': 10,
  'save_model_dir': './output/db_mv3/',
  'save_epoch_step': 1200,
  'eval_batch_step': [0, 2000],
  'cal_metric_during_train': False,
  'pretrained_model': './pretrain_models/MobileNetV3_large_x0_5_pretrained',
  'checkpoints': None,
  'save_inference_dir': None,
  'use_visualdl': False,
  'infer_img': 'doc/imgs_en/img_10.jpg',
  'save_res_path': './output/det_db/predicts_db.txt'},
 'Architecture': {'model_type': 'det',
  'algorithm': 'DB',
  'Transform': None,
  'Backbone': {'name': 'MobileNetV3', 'scale': 0.5, 'model_name': 'large'},
  'Neck': {'name': 'DBFPN', 'out_channels': 256},
  'Head': {'name': 'DBHead', 'k': 50}},
 'Loss': {'name': 'DBLoss',
  'balance_loss': True,
  'main_loss_type': 'DiceLoss',
  'alpha': 5,
  'beta': 10,
  'ohem_ratio': 3},
 'Optimizer': {'name': 'Adam',
  'beta1': 0.9,
  'beta2': 0.999,
  'lr': {'learning_rate': 0.001},
  'regularizer': {'name': 'L2', '

In [7]:
# 设置设备
device = torch.device('cuda:2' if global_config['use_gpu'] and torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda:2


In [8]:
# 初始化 TensorBoard
writer = SummaryWriter(log_dir='./output/tensorboard')

In [23]:
# 提取关键参数
arch = config['Architecture']
train_cfg = config['Train']
global_cfg = config['Global']

In [24]:
train_cfg

{'dataset': {'name': 'SimpleDataSet',
  'data_dir': './train_data/icdar2015/text_localization/',
  'label_file_list': ['./train_data/icdar2015/text_localization/train_icdar2015_label.txt'],
  'ratio_list': [1.0],
  'transforms': [{'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
   {'DetLabelEncode': None},
   {'IaaAugment': {'augmenter_args': [{'type': 'Fliplr', 'args': {'p': 0.5}},
      {'type': 'Affine', 'args': {'rotate': [-10, 10]}},
      {'type': 'Resize', 'args': {'size': [0.5, 3]}}]}},
   {'EastRandomCropData': {'size': [640, 640],
     'max_tries': 50,
     'keep_ratio': True}},
   {'MakeBorderMap': {'shrink_ratio': 0.4,
     'thresh_min': 0.3,
     'thresh_max': 0.7}},
   {'MakeShrinkMap': {'shrink_ratio': 0.4, 'min_text_size': 8}},
   {'NormalizeImage': {'scale': '1./255.',
     'mean': [0.485, 0.456, 0.406],
     'std': [0.229, 0.224, 0.225],
     'order': 'hwc'}},
   {'ToCHWImage': None},
   {'KeepKeys': {'keep_keys': ['image',
      'threshold_map',
      't

In [11]:
arch

{'model_type': 'det',
 'algorithm': 'DB',
 'Transform': None,
 'Backbone': {'name': 'MobileNetV3', 'scale': 0.5, 'model_name': 'large'},
 'Neck': {'name': 'DBFPN', 'out_channels': 256},
 'Head': {'name': 'DBHead', 'k': 50}}

In [14]:
# ==========================
# 2. 手动拼装模型 (拒绝工厂函数)
# ==========================
# 我们不调用 build_model，而是直接 new 对象
# 注意：这里假设你已经把 MobileNetV3, DBFPN, DBHead 这几个类定义写好了

# Step 1: 初始化骨干网络 (Backbone)
# 直接传入 YAML 里的参数
from det_mobilenet_v3 import MobileNetV3
backbone = MobileNetV3(
    scale=arch['Backbone']['scale'], 
    model_name=arch['Backbone']['model_name']
)



In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block for PyTorch"""
    def __init__(self, channel, reduction=4):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel // reduction, channel, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x)
        y = self.conv(y)
        return x * y

class DBFPN(nn.Module):
    def __init__(self, in_channels, out_channels, attention=False):
        super(DBFPN, self).__init__()
        self.out_channels = out_channels
        self.attention = attention

        # 1. 输入变换 (Inception-like 1x1 Conv)
        self.in2_conv = nn.Conv2d(in_channels=in_channels[0], out_channels=self.out_channels, kernel_size=1, bias=False)
        self.in3_conv = nn.Conv2d(in_channels=in_channels[1], out_channels=self.out_channels, kernel_size=1, bias=False)
        self.in4_conv = nn.Conv2d(in_channels=in_channels[2], out_channels=self.out_channels, kernel_size=1, bias=False)
        self.in5_conv = nn.Conv2d(in_channels=in_channels[3], out_channels=self.out_channels, kernel_size=1, bias=False)

        # 2. 注意力模块 (可选)
        if self.attention:
            self.p5_attention = SEBlock(self.out_channels)
            self.p4_attention = SEBlock(self.out_channels)
            self.p3_attention = SEBlock(self.out_channels)
            self.p2_attention = SEBlock(self.out_channels)

        # 3. 输出变换 (Final Prediction)
        self.p2_conv = nn.Conv2d(in_channels=self.out_channels, out_channels=self.out_channels // 4, kernel_size=3, padding=1, bias=False)
        self.p3_conv = nn.Conv2d(in_channels=self.out_channels, out_channels=self.out_channels // 4, kernel_size=3, padding=1, bias=False)
        self.p4_conv = nn.Conv2d(in_channels=self.out_channels, out_channels=self.out_channels // 4, kernel_size=3, padding=1, bias=False)
        self.p5_conv = nn.Conv2d(in_channels=self.out_channels, out_channels=self.out_channels // 4, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        c2, c3, c4, c5 = x

        # 1. 统一通道数
        in2 = self.in2_conv(c2)
        in3 = self.in3_conv(c3)
        in4 = self.in4_conv(c4)
        in5 = self.in5_conv(c5)

        # 2. 自顶向下路径 (Top-down pathway)
        out5 = in5
        out4 = in4 + F.interpolate(out5, scale_factor=2, mode='nearest')
        out3 = in3 + F.interpolate(out4, scale_factor=2, mode='nearest')
        out2 = in2 + F.interpolate(out3, scale_factor=2, mode='nearest')

        # 3. 注意力处理 (如果开启)
        if self.attention:
            out5 = self.p5_attention(out5)
            out4 = self.p4_attention(out4)
            out3 = self.p3_attention(out3)
            out2 = self.p2_attention(out2)

        # 4. 输出卷积处理
        p5 = self.p5_conv(out5)
        p4 = self.p4_conv(out4)
        p3 = self.p3_conv(out3)
        p2 = self.p2_conv(out2)

        # 5. 上采样并拼接
        p3 = F.interpolate(p3, scale_factor=2, mode='nearest')
        p4 = F.interpolate(p4, scale_factor=4, mode='nearest')
        p5 = F.interpolate(p5, scale_factor=8, mode='nearest')

        fuse = torch.cat([p2, p3, p4, p5], dim=1)
        
        return fuse

# Step 2: 初始化特征金字塔 (Neck)
neck = DBFPN(
    in_channels=[16, 24, 40, 112, 960], # 这是 MobileNetV3-Large 的输出通道，根据实际情况调整
    out_channels=arch['Neck']['out_channels']
)


In [16]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init


def get_bias_attr(fan_out):
    # 对应 Paddle 的 get_bias_attr 函数
    # stdv = 1.0 / math.sqrt(k * 1.0) 其中 k 是 fan_out
    stdv = 1.0 / math.sqrt(fan_out)
    # 返回一个初始化函数，用于在层创建后调用
    def init_bias(m):
        m.bias.data.uniform_(-stdv, stdv)
    return init_bias


class Head(nn.Module):
    def __init__(self, in_channels, name_list):
        super(Head, self).__init__()
        
        # 1. Conv2D -> Conv2d
        #    bias_attr=False 对应 bias=False
        #    weight_attr=ParamAttr() 对应默认初始化，PyTorch Conv2d 默认就是Kaiming Uniform
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels // 4,
            kernel_size=3,
            padding=1,
            bias=False) # PyTorch 默认有 bias，需显式关闭
        
        # 2. BatchNorm
        #    num_channels -> num_features
        #    param_attr (weight) 初始化为 Constant(1.0)
        #    bias_attr (bias) 初始化为 Constant(1e-4)
        #    act='relu' -> nn.ReLU(inplace=True)
        self.conv_bn1 = nn.Sequential(
            nn.BatchNorm2d(
                num_features=in_channels // 4,
                # PyTorch BatchNorm 的 weight 和 bias 默认就是 1 和 0
                # 我们在 reset_parameters 或 __init__ 后手动修改
            ),
            nn.ReLU(inplace=True)
        )
        
        # 3. Conv2DTranspose -> ConvTranspose2d
        #    weight_attr 使用 KaimingUniform -> 使用 kaiming_uniform_ 初始化
        #    bias_attr 使用 get_bias_attr -> 需在 reset 中调用
        self.conv2 = nn.ConvTranspose2d(
            in_channels=in_channels // 4,
            out_channels=in_channels // 4,
            kernel_size=2,
            stride=2,
            bias=True # 默认为 True，这里显式写出
        )
        
        self.conv_bn2 = nn.Sequential(
            nn.BatchNorm2d(num_features=in_channels // 4),
            nn.ReLU(inplace=True)
        )
        
        # 4. 最后的 Transpose
        self.conv3 = nn.ConvTranspose2d(
            in_channels=in_channels // 4,
            out_channels=1,
            kernel_size=2,
            stride=2,
            bias=True
        )

        # 5. 手动进行权重初始化 (替代 Paddle 的 weight_attr / bias_attr)
        self._initialize_weights(in_channels)

    def _initialize_weights(self, in_channels):
        # 初始化 conv1 (BN 通常不需要手动初始化 weight/bias，但为了严格对齐)
        init.constant_(self.conv_bn1[0].weight, 1.0)
        init.constant_(self.conv_bn1[0].bias, 1e-4)
        
        # 初始化 conv2 (Transposed)
        # PyTorch 默认初始化通常是 Kaiming Uniform，这里显式调用以确保
        # bias 初始化
        get_bias_attr(in_channels // 4)(self.conv2)
        
        # 初始化 conv_bn2
        init.constant_(self.conv_bn2[0].weight, 1.0)
        init.constant_(self.conv_bn2[0].bias, 1e-4)
        
        # 初始化 conv3 (Transposed)
        get_bias_attr(in_channels // 4)(self.conv3)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv_bn1(x)
        x = self.conv2(x)
        x = self.conv_bn2(x)
        x = self.conv3(x)
        x = torch.sigmoid(x) # F.sigmoid 已弃用，推荐直接使用 torch.sigmoid
        return x


class DBHead(nn.Module):
    """
    Differentiable Binarization (DB) for text detection:
        see https://arxiv.org/abs/1911.08947
    args:
        params(dict): super parameters for build DB network
    """

    def __init__(self, in_channels, k=50, **kwargs):
        super(DBHead, self).__init__()
        self.k = k
        # 注意：PyTorch 不需要 name_list 来定义参数名，参数名由 Module 的属性名自动确定
        # 这里保留变量但实际代码中不使用
        binarize_name_list = [
            'conv2d_56', 'batch_norm_47', 'conv2d_transpose_0', 'batch_norm_48',
            'conv2d_transpose_1', 'binarize'
        ]
        thresh_name_list = [
            'conv2d_57', 'batch_norm_49', 'conv2d_transpose_2', 'batch_norm_50',
            'conv2d_transpose_3', 'thresh'
        ]
        
        # 直接实例化 Head
        self.binarize = Head(in_channels, binarize_name_list)
        self.thresh = Head(in_channels, thresh_name_list)

    def step_function(self, x, y):
        # 对应 paddle.reciprocal(1 + paddle.exp(-self.k * (x - y)))
        # 即 1 / (1 + exp(-k*(x-y)))
        return torch.reciprocal(1 + torch.exp(-self.k * (x - y)))

    def forward(self, x, targets=None):
        shrink_maps = self.binarize(x)
        
        # PyTorch 中通常用 'training' 属性判断，或者显式传参
        # 这里逻辑保持一致
        if not self.training:
            return {'maps': shrink_maps}

        threshold_maps = self.thresh(x)
        binary_maps = self.step_function(shrink_maps, threshold_maps)
        
        # 拼接 dim=1 (Channel 维度)
        y = torch.cat([shrink_maps, threshold_maps, binary_maps], dim=1)
        return {'maps': y}


# Step 3: 初始化预测头 (Head)
head = DBHead(
    in_channels=arch['Neck']['out_channels'],
    k=arch['Head']['k']
)


In [18]:
# Step 4: 组合成最终模型
# 这里我们写一个最简单的容器，或者直接顺序执行
class DBNet(nn.Module):
    def __init__(self, backbone, neck, head):
        super().__init__()
        self.backbone = backbone
        self.neck = neck
        self.head = head

    def forward(self, x):
        # 1. 骨干提取特征
        x = self.backbone(x) 
        # 2. 特征融合
        x = self.neck(x) 
        # 3. 输出预测结果
        x = self.head(x) 
        return x

# 实例化模型
model = DBNet(backbone, neck, head)
print(model)

DBNet(
  (backbone): MobileNetV3(
    (conv): ConvBNLayer(
      (conv): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (stages): Sequential(
      (stage_0): Sequential(
        (0): ResidualUnit(
          (expand_conv): ConvBNLayer(
            (conv): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (bottleneck_conv): ConvBNLayer(
            (conv): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=8, bias=False)
            (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (linear_conv): ConvBNLayer(
            (conv): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True,

In [26]:
# ==========================
# 3. 直接构建训练组件
# ==========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 直接读取配置里的超参数，不经过任何中间层
optimizer = Adam(
    model.parameters(), 
    lr=cfg['Optimizer']['lr']['learning_rate'],
    weight_decay=cfg['Optimizer']['regularizer']['factor']
)


In [27]:
cfg

{'Global': {'use_gpu': True,
  'epoch_num': 1200,
  'log_smooth_window': 20,
  'print_batch_step': 10,
  'save_model_dir': './output/db_mv3/',
  'save_epoch_step': 1200,
  'eval_batch_step': [0, 2000],
  'cal_metric_during_train': False,
  'pretrained_model': './pretrain_models/MobileNetV3_large_x0_5_pretrained',
  'checkpoints': None,
  'save_inference_dir': None,
  'use_visualdl': False,
  'infer_img': 'doc/imgs_en/img_10.jpg',
  'save_res_path': './output/det_db/predicts_db.txt'},
 'Architecture': {'model_type': 'det',
  'algorithm': 'DB',
  'Transform': None,
  'Backbone': {'name': 'MobileNetV3', 'scale': 0.5, 'model_name': 'large'},
  'Neck': {'name': 'DBFPN', 'out_channels': 256},
  'Head': {'name': 'DBHead', 'k': 50}},
 'Loss': {'name': 'DBLoss',
  'balance_loss': True,
  'main_loss_type': 'DiceLoss',
  'alpha': 5,
  'beta': 10,
  'ohem_ratio': 3},
 'Optimizer': {'name': 'Adam',
  'beta1': 0.9,
  'beta2': 0.999,
  'lr': {'learning_rate': 0.001},
  'regularizer': {'name': 'L2', '

In [32]:
# ==========================
# 4. 数据加载 (保持最简)
# ==========================
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

def custom_collate_fn(batch):
    """
    自定义 Collate 函数，用于处理不同数量检测框的样本。
    它将图片堆叠成 Batch，但将 boxes 和 texts 保留为列表。
    """
    images = []
    boxes_list = []
    texts_list = []
    paths = []

    for item in batch:
        images.append(item['image'])
        boxes_list.append(item['boxes']) # 保持为 [N, 8] 的 Tensor
        texts_list.append(item['texts']) # 保持为 List
        paths.append(item['path'])

    # 将图片堆叠成一个大的 Tensor [Batch_Size, C, H, W]
    # 注意：这里假设所有图片已经被 transform 调整到了相同大小 (img_size)
    images = torch.stack(images, dim=0) 

    return {
        'image': images,     # Tensor [B, C, H, W]
        'boxes': boxes_list, # List[Tensor], 长度为 B，每个元素形状为 [N_i, 8]
        'texts': texts_list, # List[List[str]], 长度为 B
        'path': paths        # List[str], 长度为 B
    }

class ICDAR2015_JsonDataset(Dataset):
    def __init__(self, img_dir, label_file, is_training=True, img_size=640):
        """
        Args:
            img_dir (str): 图片根目录
            label_file (str): 标注文件路径 (txt，每行是一个 JSON 字符串)
            is_training (bool): 是否训练模式
            img_size (int): 图像尺寸
        """
        super(ICDAR2015_JsonDataset, self).__init__()
        self.img_dir = img_dir
        self.label_file = label_file
        self.is_training = is_training
        self.img_size = img_size
        
        # 1. 读取数据列表
        self.data_lines = self.get_image_info_list(label_file)
        
        # 2. 定义数据增强
        if is_training:
            self.transform = T.Compose([
                T.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
                T.RandomHorizontalFlip(p=0.5),
                T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
                T.ToTensor(),
                T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            self.transform = T.Compose([
                T.Resize(size=(img_size, img_size)),
                T.ToTensor(),
                T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])

    def get_image_info_list(self, file_path):
        """读取标注文件，返回列表"""
        data_lines = []
        with open(file_path, "r", encoding='utf-8') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip("\n")
                if line:
                    data_lines.append(line)
        return data_lines

    def __getitem__(self, idx):
        data_line = self.data_lines[idx]
        
        try:
            # --- 关键修改：解析 JSON 格式 ---
            # 原始格式: "path/to/img.jpg\t[{'transcription': '...', 'points': ...}]"
            # 我们需要先分割制表符，然后解析后半部分的 JSON
                        
            img_path_part, json_str = data_line.split('\t', 1) # 只分割第一个制表符
            
            # 解析 JSON 字符串
            annotations = json.loads(json_str)
            
            img_path = os.path.join(self.img_dir, img_path_part)
            
            # 检查文件是否存在
            if not os.path.exists(img_path):
                raise Exception(f"{img_path} does not exist!")
            
            # 3. 使用 PIL 读取图片
            image = Image.open(img_path).convert('RGB')
            orig_w, orig_h = image.size # 记录原始尺寸，用于还原坐标
            
            # 4. 提取标注数据 (Boxes 和 Texts)
            boxes = []
            texts = []
            
            for ann in annotations:
                transcription = ann.get('transcription', '')
                points = ann.get('points', [])
                
                # 过滤 "###" (Don't care)
                if transcription == "###":
                    continue
                
                # 确保有4个点
                if len(points) == 4:
                    # points 是 [[x1, y1], [x2, y2], ...]
                    # 展平为 [x1, y1, x2, y2, x3, y3, x4, y4]
                    box = [coord for point in points for coord in point]
                    boxes.append(box)
                    texts.append(transcription)
            
            # 如果没有有效的框，返回空列表
            if len(boxes) == 0:
                boxes = np.zeros((0, 8), dtype=np.float32)
                texts = []
            else:
                boxes = np.array(boxes, dtype=np.float32)
            
            # 5. 应用数据增强
            # 注意：TorchVision 的 transforms 不会自动调整 boxes 坐标
            # 如果需要训练，通常需要编写自定义 transform 或者使用 albumentations
            # 这里为了演示，仅对图像进行变换
            image = self.transform(image)
            
            # 构造返回数据
            data = {
                'image': image, # 已经是 Tensor [C, H, W]
                'boxes': boxes, # 应该是 Tensor [N, 8], N 是该图片的框数，每张图 N 可能不同
                'texts': texts, # List of strings
                'path': img_path
            }
            return data
            
        except Exception as e:
            print(f"Error loading {data_line}: {str(e)}")
            # 简单的容错处理
            return self.__getitem__((idx + 1) % len(self.data_lines))

    def __len__(self):
        return len(self.data_lines)


label_path = "icdar2015/text_localization/train_icdar2015_label.txt"
img_dir = "icdar2015/text_localization/" 
train_dataset = ICDAR2015_JsonDataset(img_dir=img_dir, label_file=label_path, is_training=True)
train_dataset

In [34]:

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=cfg['Train']['loader']['batch_size_per_card'],
    shuffle=True
)

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BalanceLoss(nn.Module):
    def __init__(self,
                 balance_loss=True,
                 main_loss_type='DiceLoss',
                 negative_ratio=3,
                 return_origin=False,
                 eps=1e-6,
                 **kwargs):
        """
        The BalanceLoss for Differentiable Binarization text detection
        args:
            balance_loss (bool): whether balance loss or not, default is True
            main_loss_type (str): can only be one of ['CrossEntropy','DiceLoss',
                'Euclidean','BCELoss', 'MaskL1Loss'], default is  'DiceLoss'.
            negative_ratio (int|float): float, default is 3.
            return_origin (bool): whether return unbalanced loss or not, default is False.
            eps (float): default is 1e-6.
        """
        super(BalanceLoss, self).__init__()
        self.balance_loss = balance_loss
        self.main_loss_type = main_loss_type
        self.negative_ratio = negative_ratio
        self.return_origin = return_origin
        self.eps = eps

        if self.main_loss_type == "CrossEntropy":
            self.loss = nn.CrossEntropyLoss()
        elif self.main_loss_type == "Euclidean":
            self.loss = nn.MSELoss()
        elif self.main_loss_type == "DiceLoss":
            self.loss = DiceLoss(self.eps)
        elif self.main_loss_type == "BCELoss":
            self.loss = BCELoss(reduction='none')
        elif self.main_loss_type == "MaskL1Loss":
            self.loss = MaskL1Loss(self.eps)
        else:
            loss_type = ['CrossEntropy', 'DiceLoss', 'Euclidean', 'BCELoss', 'MaskL1Loss']
            raise Exception(
                "main_loss_type in BalanceLoss() can only be one of {}".format(loss_type))

    def forward(self, pred, gt, mask=None):
        """
        The BalanceLoss for Differentiable Binarization text detection
        args:
            pred (tensor): predicted feature maps.
            gt (tensor): ground truth feature maps.
            mask (tensor): masked maps.
        return: (tensor) balanced loss
        """
        positive = gt * mask
        negative = (1 - gt) * mask

        positive_count = int(positive.sum().item())
        negative_count = int(min(negative.sum().item(), positive_count * self.negative_ratio))
        
        # Calculate the main loss (should be per-pixel loss since reduction='none' for BCE/MaskL1)
        loss = self.loss(pred, gt, mask=mask)

        if not self.balance_loss:
            return loss

        positive_loss = positive * loss
        negative_loss = negative * loss
        
        # Reshape to 1D for sorting/topk
        negative_loss = negative_loss.view(-1)
        
        if negative_count > 0:
            # Use topk to get the hardest negative samples (largest loss)
            # In PyTorch, we can directly use topk
            negative_loss = negative_loss.topk(negative_count)[0] # [0] is values, [1] is indices
            balance_loss = (positive_loss.sum() + negative_loss.sum()) / (positive_count + negative_count + self.eps)
        else:
            balance_loss = positive_loss.sum() / (positive_count + self.eps)
            
        if self.return_origin:
            return balance_loss, loss

        return balance_loss


class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, pred, gt, mask, weights=None):
        """
        DiceLoss function.
        """
        assert pred.shape == gt.shape
        assert pred.shape == mask.shape
        
        if weights is not None:
            assert weights.shape == mask.shape
            mask = weights * mask
            
        intersection = torch.sum(pred * gt * mask)
        union = torch.sum(pred * mask) + torch.sum(gt * mask) + self.eps
        loss = 1 - 2.0 * intersection / union
        
        # Ensure loss is within [0, 1]
        assert loss <= 1, f"Dice loss should be <= 1, but got {loss}"
        return loss


class MaskL1Loss(nn.Module):
    def __init__(self, eps=1e-6):
        super(MaskL1Loss, self).__init__()
        self.eps = eps

    def forward(self, pred, gt, mask):
        """
        Mask L1 Loss
        """
        # Calculate L1 loss only on masked regions and average
        loss = (torch.abs(pred - gt) * mask).sum() / (mask.sum() + self.eps)
        # Note: Removed the redundant torch.mean(loss) as loss is already a scalar
        return loss


class BCELoss(nn.Module):
    def __init__(self, reduction='mean'):
        super(BCELoss, self).__init__()
        self.reduction = reduction

    def forward(self, input, label, mask=None, weight=None, name=None):
        # Use F.binary_cross_entropy directly
        # Note: If you need to apply mask later in BalanceLoss, set reduction='none'
        loss = F.binary_cross_entropy(input, label, reduction=self.reduction)
        return loss

In [43]:
class DBLoss(nn.Module):
    """
    Differentiable Binarization (DB) Loss Function in PyTorch
    args:
        balance_loss (bool): whether balance loss or not, default is True
        main_loss_type (str): can only be one of ['CrossEntropy','DiceLoss',
            'Euclidean','BCELoss', 'MaskL1Loss'], default is  'DiceLoss'.
        alpha (float): the weight of loss_shrink_maps, default is 5.
        beta (float): the weight of loss_threshold_maps, default is 10.
        ohem_ratio (int|float): float, default is 3.
        eps (float): default is 1e-6.
    """

    def __init__(self,
                 balance_loss=True,
                 main_loss_type='DiceLoss',
                 alpha=5,
                 beta=10,
                 ohem_ratio=3,
                 eps=1e-6,
                 **kwargs):
        super(DBLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        
        # 初始化各个子损失函数
        self.dice_loss = DiceLoss(eps=eps)
        self.l1_loss = MaskL1Loss(eps=eps)
        
        # BalanceLoss 用于计算 shrink_maps 的损失 (通常使用 BCE + OHEM)
        self.bce_loss = BalanceLoss(
            balance_loss=balance_loss,
            main_loss_type=main_loss_type,
            negative_ratio=ohem_ratio)

    def forward(self, predicts, labels):
        """
        predicts: dict, contains 'maps' key, value is tensor of shape [N, 3, H, W]
                  maps[:, 0, :, :] -> shrink_maps (probability map)
                  maps[:, 1, :, :] -> threshold_maps (threshold map)
                  maps[:, 2, :, :] -> binary_maps (approximate step map)
        labels: list/tuple, contains [image, threshold_map, threshold_mask, shrink_map, shrink_mask]
                or similar structure depending on your dataset implementation.
                Here we assume labels[1:] corresponds to [threshold_map, threshold_mask, shrink_map, shrink_mask]
        """
        # 1. 提取预测结果
        predict_maps = predicts['maps']
        
        # 2. 提取标签
        # 注意：这里假设 labels 是一个列表，且顺序与 PaddleOCR 数据集返回的一致
        # labels[0] 是 image (不需要)
        # labels[1] 是 label_threshold_map
        # labels[2] 是 label_threshold_mask
        # labels[3] 是 label_shrink_map
        # labels[4] 是 label_shrink_mask
        label_threshold_map = labels[1]
        label_threshold_mask = labels[2]
        label_shrink_map = labels[3]
        label_shrink_mask = labels[4]

        # 3. 分离预测通道
        shrink_maps = predict_maps[:, 0, :, :]      # 概率图
        threshold_maps = predict_maps[:, 1, :, :]   # 阈值图
        binary_maps = predict_maps[:, 2, :, :]      # 二值化图 (用于 Dice Loss)

        # 4. 计算各项损失
        # Shrink Maps Loss (使用 BalanceLoss 处理正负样本不平衡)
        loss_shrink_maps = self.bce_loss(shrink_maps, label_shrink_map, label_shrink_mask)
        
        # Threshold Maps Loss (使用 L1 Loss 回归阈值)
        loss_threshold_maps = self.l1_loss(threshold_maps, label_threshold_map, label_threshold_mask)
        
        # Binary Maps Loss (使用 Dice Loss 优化最终分割效果)
        loss_binary_maps = self.dice_loss(binary_maps, label_shrink_map, label_shrink_mask)

        # 5. 加权求和
        loss_shrink_maps = self.alpha * loss_shrink_maps
        loss_threshold_maps = self.beta * loss_threshold_maps

        # 6. 总损失
        loss_all = loss_shrink_maps + loss_threshold_maps + loss_binary_maps
        
        # 7. 返回字典
        losses = {
            'loss': loss_all,
            "loss_shrink_maps": loss_shrink_maps,
            "loss_threshold_maps": loss_threshold_maps,
            "loss_binary_maps": loss_binary_maps
        }
        
        return losses



In [44]:
dbloss = DBLoss(cfg['Loss'])
dbloss

DBLoss(
  (dice_loss): DiceLoss()
  (l1_loss): MaskL1Loss()
  (bce_loss): BalanceLoss(
    (loss): DiceLoss()
  )
)

In [52]:
import torchvision
large = torchvision.models.mobilenet_v3_large(pretrained=True, width_mult=1.0,  reduced_tail=False, dilated=False)
small = torchvision.models.mobilenet_v3_small(pretrained=True)
quantized = torchvision.models.quantization.mobilenet_v3_large(pretrained=True)
resnet50d = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/meijian/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:55<00:00, 1.86MB/s]


In [56]:
pretrain_path = "mobilenet_v3_large-8738ca79.pth"
# A. 读取权重文件
# map_location 确保如果从 GPU 保存的模型能在 CPU 上加载
checkpoint = torch.load(pretrain_path, map_location=device)
# print(checkpoint)
# B. 提取 state_dict
# 如果是官方 Paddle 转 PyTorch 的权重，可能需要处理 key 的名字
if 'state_dict' in checkpoint:
    print('state_dict exist')
    pretrained_dict = checkpoint['state_dict']
elif 'params' in checkpoint: # Paddle 格式
     # 这里需要简单的转换逻辑，或者假设你已经转好了
    print('params exist')
    pretrained_dict = checkpoint['params'] 
else:
    print("no exist")
    pretrained_dict = checkpoint



no exist


In [57]:
# C. 处理 Key 不匹配的问题 (常见于只加载 Backbone)
# 如果预训练模型只包含 Backbone 的权重，而你的 model 包含 Backbone+Neck+Head
# 我们需要过滤掉不匹配的 key
model_dict = model.state_dict()

# 过滤掉不在 model_dict 中的 key (比如预训练模型里有 classifier，但你的模型里没有)
# 并且过滤掉 shape 不匹配的 key
filtered_pretrained_dict = {}
for k, v in pretrained_dict.items():
    # 如果是 Paddle 转的权重，key 可能带有 'backbone.' 前缀，需要去除或替换
    # 这里假设你的 MobileNetV3 类直接就是 backbone，没有前缀
    # 如果 key 不匹配，你可能需要 k.replace('module.', '') 之类的操作
    
    if k in model_dict and v.shape == model_dict[k].shape:
        filtered_pretrained_dict[k] = v
    else:
        print(f"Skip loading weight for: {k} (Shape mismatch or key not found)")

# D. 更新模型权重
model_dict.update(filtered_pretrained_dict)
model.load_state_dict(model_dict)

Skip loading weight for: features.0.0.weight (Shape mismatch or key not found)
Skip loading weight for: features.0.1.weight (Shape mismatch or key not found)
Skip loading weight for: features.0.1.bias (Shape mismatch or key not found)
Skip loading weight for: features.0.1.running_mean (Shape mismatch or key not found)
Skip loading weight for: features.0.1.running_var (Shape mismatch or key not found)
Skip loading weight for: features.0.1.num_batches_tracked (Shape mismatch or key not found)
Skip loading weight for: features.1.block.0.0.weight (Shape mismatch or key not found)
Skip loading weight for: features.1.block.0.1.weight (Shape mismatch or key not found)
Skip loading weight for: features.1.block.0.1.bias (Shape mismatch or key not found)
Skip loading weight for: features.1.block.0.1.running_mean (Shape mismatch or key not found)
Skip loading weight for: features.1.block.0.1.running_var (Shape mismatch or key not found)
Skip loading weight for: features.1.block.0.1.num_batches_tr

<All keys matched successfully>

In [45]:
# ==========================
# 5. 训练循环 (核心逻辑)
# ==========================
model.train()
print("开始训练...")

for epoch in range(cfg['Global']['epoch_num']):
    for idx, batch in enumerate(train_loader):
        images_tensor = batch['image'].to(device)
        
        # 前向传播 (最直观的写法)
        preds = model(images_tensor) # 一行代码搞定三层网络

        # DBLoss 需要的标签 (根据你的 SimpleDataSet 返回的 keys 调整)
        # 假设你的 dataset 返回了: ['image', 'threshold_map', 'threshold_mask', 'shrink_map', 'shrink_mask']
        labels = [
            batch['threshold_map'].to(device, non_blocking=True),
            batch['threshold_mask'].to(device, non_blocking=True),
            batch['shrink_map'].to(device, non_blocking=True),
            batch['shrink_mask'].to(device, non_blocking=True)
        ]

        # 2. 前向传播 (使用 AMP)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            preds = model(images) # 模型输出
            
            # 计算损失
            # 注意：这里传入的是 preds 字典和 labels 列表
            loss = dbloss(preds, batch)
            avg_loss = loss['loss']

        # 3. 反向传播 (使用 AMP)
        optimizer.zero_grad()
        
        avg_loss.backward()
        optimizer.step()
        optimizer.clear_grad()

        total_steps += 1

        # 4. 打印日志 & 记录 TensorBoard
        if total_steps % global_cfg['Global']['print_batch_step'] == 0:
            # 记录到 TensorBoard
            writer.add_scalar('Train/Loss_Total', loss.item(), total_steps)
            writer.add_scalar('Train/Loss_Shink', loss_dict['loss_shrink_maps'].item(), total_steps)
            writer.add_scalar('Train/Loss_Threshold', loss_dict['loss_threshold_maps'].item(), total_steps)
            
            # 打印到控制台
            print(f"Epoch: [{epoch+1}/{global_cfg['Global']['epoch_num']}] "
                  f"Step: [{total_steps}] "
                  f"Loss: {loss.item():.4f} "
                  f"(Shrink: {loss_dict['loss_shrink_maps'].item():.4f})")

    # --- 每个 Epoch 结束后的操作 ---
    
    # 5. 保存模型 (定期保存)
    if (epoch + 1) % global_cfg['Global']['save_epoch_step'] == 0:
        save_path = os.path.join(global_cfg['Global']['save_model_dir'], f"epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict()
        }, save_path)
        print(f"Saved model to {save_path}")

    epoch_end_time = time.time()
    print(f"Epoch {epoch+1} finished, time: {epoch_end_time - epoch_start_time:.2f}s")

writer.close()
print("Training finished!")

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)


KeyboardInterrupt

